<!-- # The Bayesian Finite Element Method in Inverse Problems: Pullout Test

This notebook is associated with section 3.1 of "The Bayesian Finite Element Method in Inverse Problems: a Critical Comparison between Probabilistic Models for Discretization Error" by Anne Poot, Iuri Rocha, Pierre Kerfriden and Frans van der Meer ([doi:10.48550/arXiv.2506.02815](https://doi.org/10.48550/arXiv.2506.02815)). -->
# Theory

In [ ]:
# general imports
import os
import numpy as np
import matplotlib.pyplot as plt

## Green's functions of the Laplacian

In [ ]:
def get_harmonic(k):
    l = (k * np.pi)**2

    def harmonic(x):
        return np.sqrt(2) * np.sin(k * np.pi * x)

    return l, harmonic


def get_eigenpairs(kmax, d):
    harmonics = []
    for k in range(1, kmax):
        harmonics.append(get_harmonic(k))

    if d == 1:
        eigenpairs_1d = []

        for idx, (eval1, efunc1) in enumerate(harmonics):
            def eigenfunc_1d(X, efunc1=efunc1):
                x = X[...,0]
                return efunc1(x)

            idcs = (idx + 1, )
            eigenpairs_1d.append((eval1, eigenfunc_1d, idcs))

        return eigenpairs_1d

    elif d == 2:
        eigenvals_2d = []

        cutoff = -int(-np.sqrt(4 * kmax / np.pi))

        for idx1, (eval1, _) in enumerate(harmonics[:cutoff]):
            for idx2, (eval2, _) in enumerate(harmonics[:cutoff]):
                eval_2d = eval1 + eval2
                eigenvals_2d.append((eval_2d, idx1, idx2))

        eigenvals_2d = sorted(eigenvals_2d, key=lambda x: x[0])[:kmax]
        eigenpairs_2d = []

        for eval_2d, idx1, idx2 in eigenvals_2d:
            efunc1 = harmonics[idx1][1]
            efunc2 = harmonics[idx2][1]

            def eigenfunc_2d(X, efunc1=efunc1, efunc2=efunc2):
                X = np.atleast_2d(X)
                x, y = X[...,0], X[...,1]
                return efunc1(x) * efunc2(y)

            idcs = (idx1 + 1, idx2 + 1)
            eigenpairs_2d.append((eval_2d, eigenfunc_2d, idcs))

        return eigenpairs_2d

    elif d == 3:
        eigenvals_3d = []

        cutoff = -int(-(6 * kmax / np.pi)**(1/3))

        for idx1, (eval1, _) in enumerate(harmonics[:cutoff]):
            for idx2, (eval2, _) in enumerate(harmonics[:cutoff]):
                for idx3, (eval3, _) in enumerate(harmonics[:cutoff]):
                    eval_3d = eval1 + eval2 + eval3
                    eigenvals_3d.append((eval_3d, idx1, idx2, idx3))

        eigenvals_3d = sorted(eigenvals_3d, key=lambda x: x[0])[:kmax]
        eigenpairs_3d = []

        for eval_3d, idx1, idx2, idx3 in eigenvals_3d:
            efunc1 = harmonics[idx1][1]
            efunc2 = harmonics[idx2][1]
            efunc3 = harmonics[idx3][1]

            def eigenfunc_3d(X, efunc1=efunc1, efunc2=efunc2, efunc3=efunc3):
                X = np.atleast_2d(X)
                x, y, z = X[...,0], X[...,1], X[...,2]
                return efunc1(x) * efunc2(y) * efunc3(z)

            idcs = (idx1 + 1, idx2 + 1, idx3 + 1)
            eigenpairs_3d.append((eval_3d, eigenfunc_3d, idcs))

        return eigenpairs_3d

        
def get_greensfunc(d, kmax=1000):
    eigenpairs = get_eigenpairs(kmax, d)

    def greensfunc(X1, X2):
        X1 = np.atleast_2d(X1)
        X2 = np.atleast_2d(X2)

        G = np.zeros((X1.shape[0], X2.shape[0]))
        
        for eigenval, eigenfunc, idcs in eigenpairs:
            if len(idcs) >= 2 and idcs[1] % 2 == 0:
                continue
            elif len(idcs) >= 3 and idcs[2] % 2 == 0:
                continue
                
            G += np.outer(eigenfunc(X1), eigenfunc(X2)) / eigenval

        return G

    return greensfunc

In [ ]:
# 1D eigenfunction plot
x = np.linspace(0, 1, 100)
X = np.reshape(x, (-1, 1))

plt.figure()
for eigenval, eigenfunc, _ in get_eigenpairs(20, 1):
    plt.plot(x, eigenfunc(X) / np.sqrt(eigenval))
plt.show()

# 2D eigenfunction plot
x = np.linspace(0, 1, 60)
y = np.linspace(0, 1, 50)
X, Y = np.meshgrid(x, y)

vmax = 0.0

fig, axs = plt.subplots(ncols=8, figsize=(8 * 4.8, 4.8))
for (eigenval, eigenfunc, _), ax in zip(get_eigenpairs(20, 2), axs):
    Z = eigenfunc(np.array([X, Y]).transpose(1, 2, 0)) / np.sqrt(eigenval)
    if np.max(Z) > vmax:
        vmax = np.max(Z)
    ax.contourf(X, Y, Z, levels=50, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_aspect("equal")
    ax.axis("off")
plt.show()

In [ ]:
dims = [1, 2, 3]
hmaxs = 2**np.arange(1, 8)

ndim = len(dims)
nh = len(hmaxs)

fig, axs = plt.subplots(nrows=ndim, ncols=nh, figsize=(nh * 2.4, ndim * 2.4), sharex=True)

for i, d in enumerate(dims):
    for j, hmax in enumerate(hmaxs):
        kmax = hmax**d
        
        ax = axs[i][j]
        
        greensfunc = get_greensfunc(d, kmax=kmax)

        if d == 1:
            x2 = np.linspace(0, 1, 1001)
            X2 = np.reshape(x2, (-1, 1))
            
            for x1 in np.linspace(0, 1, 11):
                ax.plot(x2, greensfunc(x1, X2).flatten())
    
        elif d == 2:
            y1 = y2 = 0.5
            x2 = np.linspace(0, 1, 1001)    
            X2 = np.array([x2, y2 * np.ones_like(x2)]).T
            
            for x1 in np.linspace(0, 1, 11):
                X1 = np.array([x1, y1])
                ax.plot(x2, greensfunc(X1, X2).flatten())

        elif d == 3:
            y1 = y2 = 0.5
            z1 = z2 = 0.5
            x2 = np.linspace(0, 1, 1001)    
            X2 = np.array([x2, y2 * np.ones_like(x2), z2 * np.ones_like(x2)]).T
            
            for x1 in np.linspace(0, 1, 11):
                X1 = np.array([x1, y1, z1])
                ax.plot(x2, greensfunc(X1, X2).flatten())

        # ax.set_aspect("equal")
        # axs[j][i].axis("off")
        ax.set_xlim((0.0, 1.0))
        ax.set_xticks([0.0, 0.5, 1.0])

        if d == 1:
            ax.set_ylim((0.0, 0.4))
            ax.set_yticks([0.0, 0.2, 0.4])
        elif d == 2:
            ax.set_ylim((0.0, 0.6))
            ax.set_yticks([0.0, 0.3, 0.6])
        elif d == 3:
            ax.set_ylim((0.0, 2.0))
            ax.set_yticks([0.0, 1.0, 2.0])

        if d != dims[-1]:
            ax.get_xaxis().set_visible(False)
        if hmax != hmaxs[0]:
            ax.get_yaxis().set_visible(False)

plt.savefig("greens-functions.pdf", bbox_inches="tight")
plt.show()